# Video Translation

Translates a video's audio to another language using Meta's [SeamlessM4T v2](https://huggingface.co/facebook/seamless-m4t-v2-large) — a direct speech-to-speech translation model.

**Setup:** Mount Google Drive to cache model weights (~9 GB) and store videos.

**Usage:** Set `VIDEO_PATH` and `TARGET_LANG`, then run all cells.

In [ ]:
# Mount Google Drive and install dependencies
from google.colab import drive
drive.mount("/content/drive")

!pip install -q transformers torch torchaudio ffmpeg-python tqdm

In [ ]:
# ── Config ──────────────────────────────────────────────────────────
VIDEO_PATH = "/content/drive/MyDrive/videos/input.mp4"  # input video
TARGET_LANG = "hin"  # target language code (see list below)
MODEL_CACHE = "/content/drive/MyDrive/models/seamless-m4t-v2"  # weights cache
SPEAKER_ID = 7  # voice ID (0–199). Try different values if the voice sounds off.

# Supported languages (subset — full list at https://huggingface.co/facebook/seamless-m4t-v2-large)
# fmt: off
LANGUAGES = {
    "hin": "Hindi",     "spa": "Spanish",   "fra": "French",
    "deu": "German",    "ita": "Italian",   "por": "Portuguese",
    "rus": "Russian",   "zho": "Chinese",   "jpn": "Japanese",
    "kor": "Korean",    "ara": "Arabic",    "tur": "Turkish",
    "pol": "Polish",    "nld": "Dutch",     "vie": "Vietnamese",
    "tha": "Thai",      "ind": "Indonesian","ben": "Bengali",
    "tam": "Tamil",     "tel": "Telugu",    "urd": "Urdu",
    "eng": "English",
}
# fmt: on

assert TARGET_LANG in LANGUAGES, f"Unsupported language: {TARGET_LANG}. Choose from: {list(LANGUAGES.keys())}"
print(f"Will translate → {LANGUAGES[TARGET_LANG]} ({TARGET_LANG}), speaker_id={SPEAKER_ID}")

In [ ]:
import os, torch, torchaudio, tempfile, shutil
from pathlib import Path
from tqdm.auto import tqdm
from transformers import AutoProcessor, SeamlessM4Tv2ForSpeechToSpeech

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# Load model (cached on Drive so subsequent runs are instant)
print("Loading model (first run downloads ~9 GB to Drive)...")
os.makedirs(MODEL_CACHE, exist_ok=True)

processor = AutoProcessor.from_pretrained(
    "facebook/seamless-m4t-v2-large", cache_dir=MODEL_CACHE
)
model = SeamlessM4Tv2ForSpeechToSpeech.from_pretrained(
    "facebook/seamless-m4t-v2-large", cache_dir=MODEL_CACHE
).to(device)

print("Model loaded.")

In [ ]:
import subprocess, ffmpeg

SAMPLE_RATE = 16_000  # model expects 16 kHz mono
CHUNK_SEC = 30        # process audio in chunks (longer = better translation context)
OUTPUT_SR = model.config.sampling_rate  # output sample rate from the vocoder


def extract_audio(video_path: str, sr: int = SAMPLE_RATE) -> torch.Tensor:
    """Extract mono audio from video at the given sample rate."""
    tmp = tempfile.mktemp(suffix=".wav")
    (
        ffmpeg.input(video_path)
        .output(tmp, ac=1, ar=sr, format="wav")
        .overwrite_output()
        .run(quiet=True)
    )
    waveform, _ = torchaudio.load(tmp)
    os.remove(tmp)
    return waveform.squeeze(0)  # (samples,)


def translate_audio(waveform: torch.Tensor, tgt_lang: str) -> torch.Tensor:
    """Translate audio waveform, processing in chunks with a progress bar."""
    chunk_samples = CHUNK_SEC * SAMPLE_RATE
    chunks = [waveform[i : i + chunk_samples] for i in range(0, len(waveform), chunk_samples)]
    translated_parts = []

    for i, chunk in enumerate(tqdm(chunks, desc="Translating", unit="chunk")):
        inputs = processor(
            audio=chunk.numpy(), sampling_rate=SAMPLE_RATE, return_tensors="pt"
        ).to(device)
        with torch.no_grad():
            output = model.generate(
                **inputs,
                tgt_lang=tgt_lang,
                speaker_id=SPEAKER_ID,
                return_intermediate_token_ids=True,
            )
        # Print intermediate text to verify translation is happening
        if hasattr(output, "sequences") and output.sequences is not None:
            tokens = output.sequences.squeeze().tolist()
            if isinstance(tokens, int):
                tokens = [tokens]
            text = processor.decode(tokens, skip_special_tokens=True)
            print(f"  Chunk {i+1}: {text[:120]}")

        waveform_out = output.waveform.cpu()
        # Remove batch dim if present, keep 1-D
        while waveform_out.dim() > 1:
            waveform_out = waveform_out.squeeze(0)
        # Trim to actual length (remove padding)
        if hasattr(output, "waveform_lengths") and output.waveform_lengths is not None:
            lengths = output.waveform_lengths.cpu()
            length = int(lengths.item()) if lengths.dim() == 0 else int(lengths[0].item())
            waveform_out = waveform_out[:length]
        translated_parts.append(waveform_out)

    return torch.cat(translated_parts)


def replace_audio(video_path: str, audio: torch.Tensor, output_path: str, sr: int = OUTPUT_SR):
    """Mux original video (no audio) with translated audio."""
    tmp_audio = tempfile.mktemp(suffix=".wav")
    torchaudio.save(tmp_audio, audio.unsqueeze(0), sr)

    video_in = ffmpeg.input(video_path)
    audio_in = ffmpeg.input(tmp_audio)
    (
        ffmpeg.output(
            video_in.video, audio_in.audio, output_path,
            vcodec="copy", acodec="aac", strict="experimental"
        )
        .overwrite_output()
        .run(quiet=True)
    )
    os.remove(tmp_audio)


print(f"Functions ready. Output sample rate: {OUTPUT_SR} Hz")

In [ ]:
# ── Translate ───────────────────────────────────────────────────────
video = Path(VIDEO_PATH)
assert video.exists(), f"Video not found: {VIDEO_PATH}"

output_path = str(video.with_stem(f"{video.stem}_{TARGET_LANG}"))

print(f"Input:  {VIDEO_PATH}")
print(f"Output: {output_path}")
print()

# Step 1: Extract audio
print("Extracting audio...")
waveform = extract_audio(VIDEO_PATH)
duration = len(waveform) / SAMPLE_RATE
print(f"Audio duration: {duration:.1f}s")

# Step 2: Translate
print(f"\nTranslating to {LANGUAGES[TARGET_LANG]}...")
translated = translate_audio(waveform, TARGET_LANG)

# Step 3: Replace audio in video
print("\nMuxing translated audio with original video...")
replace_audio(VIDEO_PATH, translated, output_path)

print(f"\n✓ Done! Translated video saved to:\n{output_path}")